In [ ]:
# ==============================================================================\
# CELL 1: TẢI DỮ LIỆU TỪ GOOGLE DRIVE (GIỮ RIÊNG TRAIN/VAL)
# ==============================================================================
!pip install gdown -q

import os
import gdown
import pandas as pd
import shutil

# 1. CẤU HÌNH THƯ MỤC
DATA_INTERIM = "data/interim"
DATA_PROCESSED = "data/processed"
os.makedirs(DATA_INTERIM, exist_ok=True)
os.makedirs(DATA_PROCESSED, exist_ok=True)

# 2. LINK FILE (ID lấy từ code cũ của bạn)
# Lưu ý: Tôi đã chuyển link view sang format tải trực tiếp của gdown
FILES = {
    # File Train Log
    "temp_train.parquet": {
        "url": "https://drive.google.com/uc?id=1qEP4UEoysTTUwLRIu8pRsyF3OIpRxMfD",
        "dest": f"{DATA_INTERIM}/train.parquet" # Đổi tên luôn thành train.parquet
    },
    # File Val Log (Giữ riêng để đánh giá)
    "temp_val.parquet": {
        "url": "https://drive.google.com/uc?id=1g6WYtdoMK-JYv7m7t3mnlD2D0t0SWbI8",
        "dest": f"{DATA_INTERIM}/val.parquet"   # Đổi tên luôn thành val.parquet
    },
    # Metadata Train (Chứa thông tin item cũ)
    "metadata.parquet": {
        "url": "https://drive.google.com/uc?id=1ELz1QRp9EEVK2g086b_NyiiZvPtmT6vb",
        "dest": f"{DATA_PROCESSED}/metadata.parquet"
    },
    # Metadata Test (Chứa thông tin item tương lai - Target)
    "metadata_test.parquet": {
        "url": "https://drive.google.com/uc?id=10aQeyaaHFKqXJ5VR03M1LH_mjJpVAbH7",
        "dest": f"{DATA_PROCESSED}/metadata_test.parquet"
    }
}

def download_and_setup():
    print(" Bắt đầu tải dữ liệu...")

    for original_name, info in FILES.items():
        output_path = info["dest"]
        url = info["url"]

        if not os.path.exists(output_path):
            print(f"Đang tải {original_name}...")
            gdown.download(url, output_path, quiet=False)
        else:
            print(f" {original_name} đã tồn tại.")

    # Kiểm tra lại file
    print("\n Kiểm tra thư mục dữ liệu:")
    if os.path.exists(DATA_INTERIM):
        print(f"   {DATA_INTERIM}/: {os.listdir(DATA_INTERIM)}")
    if os.path.exists(DATA_PROCESSED):
        print(f"   {DATA_PROCESSED}/: {os.listdir(DATA_PROCESSED)}")

    print("\n Dữ liệu đã sẵn sàng để xây dựng đồ thị!")

if __name__ == "__main__":
    download_and_setup()

In [ ]:
# ==============================================================================
# CELL 1: SETUP, MODEL & FAST TRAINING (OPTIMIZED)
# ==============================================================================
!pip install sentence-transformers scikit-learn pandas numpy tqdm matplotlib -q

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
from tqdm import tqdm
from sentence_transformers import SentenceTransformer
import ast
import os
import zipfile

# --- CẤU HÌNH TỐI ƯU TỐC ĐỘ ---
MODEL_NAME = 'paraphrase-multilingual-MiniLM-L12-v2'
EMBED_DIM = 384
HIDDEN_DIM = 128
BATCH_SIZE = 4096    # Tăng lên 4096 để chạy nhanh hơn
LR = 0.001
EPOCHS = 3           # Giảm xuống 3 epoch là đủ hội tụ với data lớn
MAX_TRAIN_SAMPLES = 300000 # Chỉ lấy 300k mẫu tốt nhất để train

# --- 1. XỬ LÝ TEXT & SBERT ---
def clean_text(val):
    if isinstance(val, (list, np.ndarray)): return ", ".join([str(x) for x in val])
    if isinstance(val, str) and val.strip().startswith('['):
        try: return ", ".join(ast.literal_eval(val))
        except: pass
    return str(val) if not pd.isna(val) else ""

def build_sbert_features(df_meta):
    print(f" [SBERT] Creating features for {len(df_meta)} items...")
    model = SentenceTransformer(MODEL_NAME)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model.to(device)

    df_meta['semantic_text'] = df_meta.apply(lambda x: (
        f"Category: {str(x.get('tv_show_category', ''))}. "
        f"Genre: {clean_text(x.get('genres_list'))}. "
        f"Actors: {clean_text(x.get('actors_list'))}. "
        f"Director: {str(x.get('director', ''))}. "
        f"Channel: {str(x.get('vsetv_id', ''))}."
    ), axis=1)

    sentences = df_meta['semantic_text'].tolist()
    # Batch encode nhanh
    embeddings = model.encode(sentences, batch_size=128, show_progress_bar=True, device=device)

    if 'str_id' not in df_meta.columns:
        df_meta['str_id'] = df_meta['tv_show_id'].astype(str).str.replace(r'\.0$', '', regex=True)

    all_ids = df_meta['str_id'].values
    item_map = {idn: i for i, idn in enumerate(all_ids)}
    idx_to_id = {i: idn for i, idn in enumerate(all_ids)}

    return embeddings, item_map, idx_to_id, df_meta

# --- 2. PREPARE DATA VỚI SAMPLING ---
def prepare_data_with_implicit_rating(logs, max_samples=MAX_TRAIN_SAMPLES):
    print(" Preparing Data & Sampling...")
    df = logs.copy()
    df['str_id'] = df['tv_show_id'].astype(str).str.replace(r'\.0$', '', regex=True)
    df = df[df['str_id'] != '0']

    # Tính Implicit Rating
    t_ref = pd.to_datetime(df["start_time_view"]).max()
    df["days_ago"] = (t_ref - pd.to_datetime(df["start_time_view"])).dt.days

    # Công thức Implicit
    df["implicit_rating"] = (df["screen_time"].clip(0, 1.5) ** 1.2) * np.exp(-0.02 * df["days_ago"])

    # Lấy threshold top 50%
    threshold = df["implicit_rating"].quantile(0.5)
    train_candidates = df[df["implicit_rating"] >= threshold]

    # --- SAMPLING QUAN TRỌNG ---
    # Nếu data quá lớn, chỉ lấy mẫu ngẫu nhiên (ưu tiên rating cao)
    if len(train_candidates) > max_samples:
        print(f"  Data quá lớn ({len(train_candidates)}). Sampling xuống {max_samples}...")
        # Lấy mẫu có trọng số: rating càng cao càng dễ được chọn
        weights = train_candidates["implicit_rating"] / train_candidates["implicit_rating"].sum()
        train_samples = train_candidates.sample(n=max_samples, weights=weights, random_state=42)
    else:
        train_samples = train_candidates.copy()

    print(f" Training Samples: {len(train_samples)}")
    return train_samples, df

# --- 3. MODEL & DATASET ---
class BPRDataset(Dataset):
    def __init__(self, df, user_map, item_map, num_items):
        # Pre-compute numpy arrays để truy xuất siêu tốc
        valid_mask = (df['user_id'].isin(user_map)) & (df['str_id'].isin(item_map))
        filtered = df[valid_mask]

        self.users = filtered['user_id'].map(user_map).values.astype(np.int32)
        self.items = filtered['str_id'].map(item_map).values.astype(np.int32)
        self.num_items = num_items

    def __len__(self):
        return len(self.users)

    def __getitem__(self, idx):
        u = self.users[idx]
        i = self.items[idx]
        j = np.random.randint(0, self.num_items)
        while j == i: j = np.random.randint(0, self.num_items)
        return u, i, j

class RecommenderNet(nn.Module):
    def __init__(self, num_users, sbert_matrix):
        super().__init__()
        self.user_embedding = nn.Embedding(num_users, HIDDEN_DIM)
        # SBERT fixed features
        self.sbert_features = nn.Parameter(torch.tensor(sbert_matrix, dtype=torch.float32), requires_grad=False)
        # Adapter network
        self.item_adapter = nn.Sequential(
            nn.Linear(EMBED_DIM, 256),
            nn.ReLU(),
            nn.Linear(256, HIDDEN_DIM)
        )
        nn.init.normal_(self.user_embedding.weight, std=0.01)

    def forward(self, u, i, j):
        u_emb = self.user_embedding(u)
        i_emb = self.item_adapter(self.sbert_features[i])
        j_emb = self.item_adapter(self.sbert_features[j])
        return (u_emb * i_emb).sum(1), (u_emb * j_emb).sum(1)

    def get_user_vec(self, u_idx):
        return self.user_embedding(u_idx)

    def get_all_items(self):
        return self.item_adapter(self.sbert_features)

# --- 4. FAST TRAINING LOOP ---
def run_training(train_df, sbert_matrix, item_map, phase_name="Phase"):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    user_ids = train_df['user_id'].unique()
    user_map = {u: i for i, u in enumerate(user_ids)}
    idx_to_user = {i: u for i, u in enumerate(user_ids)}

    dataset = BPRDataset(train_df, user_map, item_map, len(sbert_matrix))
    # Num_workers=2 để load data đa luồng
    dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)

    model = RecommenderNet(len(user_ids), sbert_matrix).to(device)
    optimizer = optim.Adam(model.parameters(), lr=LR)

    loss_history = []
    print(f"\n [{phase_name}] Fast Training ({EPOCHS} Epochs, Batch={BATCH_SIZE})...")

    model.train()
    for epoch in range(EPOCHS):
        total_loss = 0
        pbar = tqdm(dataloader, desc=f"Epoch {epoch+1}")
        for u, i, j in pbar:
            u, i, j = u.to(device, non_blocking=True), i.to(device, non_blocking=True), j.to(device, non_blocking=True)

            optimizer.zero_grad()
            pos, neg = model(u, i, j)
            loss = -torch.mean(torch.nn.functional.logsigmoid(pos - neg))
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            pbar.set_postfix({'loss': loss.item()})

        avg_loss = total_loss / len(dataloader)
        loss_history.append(avg_loss)
        print(f"   -> Avg Loss: {avg_loss:.4f}")

    return model, loss_history, user_map, idx_to_user

In [ ]:
# ==============================================================================
# CELL 2: PHASE 1 - MEMORY SAFE MODE (FIX TRÀN RAM)
# ==============================================================================
import matplotlib.pyplot as plt
import gc # Thư viện dọn rác bộ nhớ
from torch.cuda.amp import autocast, GradScaler

# Hàm train tiết kiệm bộ nhớ
def train_phase_1_memory_safe(train_samples, sbert_emb, item_map):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    user_ids = train_samples['user_id'].unique()
    user_map = {u: i for i, u in enumerate(user_ids)}

    # --- FIX 1: TẮT MULTI-PROCESSING ĐỂ TIẾT KIỆM RAM ---
    dataset = BPRDataset(train_samples, user_map, item_map, len(sbert_emb))
    dataloader = DataLoader(dataset, batch_size=2048, shuffle=True, num_workers=0, pin_memory=False)

    model = RecommenderNet(len(user_ids), sbert_emb).to(device)
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    scaler = GradScaler()

    loss_history = []

    print(f"\n [Memory Safe] Training {EPOCHS} Epochs (Batch=2048)...")
    model.train()

    for epoch in range(EPOCHS):
        pbar = tqdm(dataloader, desc=f"Ep {epoch+1}")
        for u, i, j in pbar:
            u, i, j = u.to(device), i.to(device), j.to(device)
            optimizer.zero_grad()
            with autocast():
                pos, neg = model(u, i, j)
                loss = -torch.mean(torch.nn.functional.logsigmoid(pos - neg))

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            loss_history.append(loss.item())
            pbar.set_postfix({'loss': loss.item()})

    return model, loss_history, user_map

def phase_1_evaluation():
    # 1. Load Data
    print(" Loading Data...")
    df_train = pd.read_parquet("data/interim/train.parquet")

    # --- FIX 2: CHUẨN BỊ DATA TRƯỚC RỒI XÓA BẢN GỐC ---
    # Dùng hàm prepare_data_fast (nếu có) hoặc prepare_data_with_implicit_rating
    try:
        train_samples, _ = prepare_data_fast(df_train, max_samples=300000)
    except NameError:
        train_samples, _ = prepare_data_with_implicit_rating(df_train, max_samples=300000)

    # XÓA NGAY dataframe gốc để giải phóng RAM
    del df_train
    gc.collect()
    print("   -> Đã giải phóng RAM dữ liệu gốc.")

    # 2. Load Metadata & Build Features
    df_meta = pd.read_parquet("data/processed/metadata.parquet")
    sbert_emb, item_map, idx2id, _ = build_sbert_features(df_meta)

    del df_meta # Xóa tiếp metadata gốc
    gc.collect()

    # 3. Training
    model, losses, user_map = train_phase_1_memory_safe(train_samples, sbert_emb, item_map)

    # 4. Vẽ biểu đồ
    plt.figure(figsize=(10, 4))
    plt.plot(losses, color='#2ca02c', linewidth=1)
    plt.title("Training Loss (Memory Safe Mode)")
    plt.xlabel("Steps"); plt.ylabel("Loss")
    plt.grid(True, alpha=0.5); plt.show()

    # 5. Evaluation
    print("\n Evaluating...")
    # Load lại val set (chỉ load lúc cần dùng)
    df_val = pd.read_parquet("data/interim/val.parquet")

    model.eval()
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    with torch.no_grad():
        trained_items = torch.nn.functional.normalize(model.get_all_items(), p=2, dim=1)

    val_users = [u for u in df_val['user_id'].unique() if u in user_map]
    ground_truth = df_val[df_val['user_id'].isin(val_users)].groupby('user_id')['tv_show_id'].apply(lambda x: set(str(i).replace('.0','') for i in x)).to_dict()

    del df_val # Xóa val set sau khi lấy ground truth
    gc.collect()

    k=5; maps, recalls, ndcgs = [], [], []
    eval_u_indices = [user_map[u] for u in val_users]

    # Batch eval nhỏ (500) để an toàn cho RAM
    for i in range(0, len(eval_u_indices), 500):
        batch_idx = eval_u_indices[i : i+500]
        u_tensor = torch.tensor(batch_idx).to(device)
        with torch.no_grad():
            u_vec = torch.nn.functional.normalize(model.get_user_vec(u_tensor), p=2, dim=1)
            _, topk = torch.topk(torch.matmul(u_vec, trained_items.t()), k=k, dim=1)
            topk = topk.cpu().numpy()

        for idx, p_indices in enumerate(topk):
            uid = val_users[i + idx]
            true_items = ground_truth.get(uid, set())
            if not true_items: continue
            pred_items = [idx2id[pix] for pix in p_indices]

            hits = 0; ap = 0; dcg = 0; idcg = 0
            for r, item in enumerate(pred_items):
                if item in true_items:
                    hits += 1; ap += hits/(r+1); dcg += 1.0/np.log2(r+2)
            for r in range(min(len(true_items), k)): idcg += 1.0/np.log2(r+2)

            recalls.append(hits/len(true_items))
            maps.append(ap/min(len(true_items), k) if true_items else 0)
            ndcgs.append(dcg/idcg if idcg > 0 else 0)

    print(f" RESULTS: MAP@5={np.mean(maps):.4f} | Recall@5={np.mean(recalls):.4f} | NDCG@5={np.mean(ndcgs):.4f}")

if __name__ == "__main__":
    phase_1_evaluation()

In [ ]:
# ==============================================================================
# CELL 3: PHASE 2 - PRODUCTION & SUBMISSION
# ==============================================================================
def phase_2_production():
    print("\n PHASE 2: Production (Full Training -> Submission)...")

    # 1. Load All Data
    df_tr = pd.read_parquet("data/interim/train.parquet")
    df_va = pd.read_parquet("data/interim/val.parquet")
    df_full = pd.concat([df_tr, df_va], ignore_index=True)

    # Metadata gộp (Train + Test metadata)
    df_meta_tr = pd.read_parquet("data/processed/metadata.parquet")
    df_meta_te = pd.read_parquet("data/processed/metadata_test.parquet")
    df_meta_te['is_future'] = True
    df_meta_tr['is_future'] = False
    df_meta_full = pd.concat([df_meta_tr, df_meta_te], ignore_index=True).drop_duplicates('tv_show_id', keep='last')

    # 2. Build SBERT Full
    sbert_emb, item_map, idx2id, df_meta_proc = build_sbert_features(df_meta_full)

    # 3. Prepare Full Training Data
    train_samples, full_logs = prepare_data_with_implicit_rating(df_full)

    # 4. Training Full Model
    model, _, user_map, idx2user = run_training(train_samples, sbert_emb, item_map, phase_name="Phase 2 (Full)")

    # 5. Recommendation Logic
    print(" Generating Candidates for Future Items...")
    model.eval()
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Xác định Target Items (Future)
    # Lấy những item có cờ is_future=True VÀ có trong item_map
    future_ids = df_meta_full[df_meta_full['is_future'] == True]['str_id'].unique()
    target_indices = [item_map[i] for i in future_ids if i in item_map]

    if not target_indices:
        print(" Không tìm thấy future items khớp với map."); return

    # Lấy vector item đã học
    with torch.no_grad():
        all_item_vecs = model.get_all_items() # (N_all, 128)
        target_vecs = all_item_vecs[target_indices] # (N_target, 128)
        target_vecs = torch.nn.functional.normalize(target_vecs, p=2, dim=1)

    # Popularity Fallback (Tính trên Full Log)
    pop_series = full_logs['str_id'].value_counts()
    def get_pop_score(idx):
        original_id = idx2id[idx]
        return np.log1p(pop_series.get(original_id, 0))

    target_pops = torch.tensor([get_pop_score(i) for i in target_indices], device=device).unsqueeze(0)

    # Dự đoán cho tất cả User có trong Log
    # Batch processing
    all_users = list(user_map.keys())
    user_batches = [all_users[i:i+1000] for i in range(0, len(all_users), 1000)]

    rows = []

    for batch_users in tqdm(user_batches, desc="Predicting"):
        batch_indices = [user_map[u] for u in batch_users]
        u_tensor = torch.tensor(batch_indices).to(device)

        with torch.no_grad():
            u_vecs = model.get_user_vec(u_tensor)
            u_vecs = torch.nn.functional.normalize(u_vecs, p=2, dim=1)

            # Score = Sim * (1 + 0.1 * Pop)
            sims = torch.matmul(u_vecs, target_vecs.t())
            final_scores = sims * (1 + 0.1 * target_pops)

            _, top5_idx = torch.topk(final_scores, k=5, dim=1)
            top5_idx = top5_idx.cpu().numpy()

        for i, u_id in enumerate(batch_users):
            # Map index target -> original ID
            # Lưu ý: top5_idx trả về index trong mảng target_vecs, cần map về target_indices rồi về original ID
            pred_ids = [idx2id[target_indices[local_idx]] for local_idx in top5_idx[i]]
            rows.append({
                "user_id": str(u_id).replace('.0',''),
                "tv_show_id": " ".join(pred_ids)
            })

    # 6. Save Submission
    print(" Saving Submission...")
    sub_df = pd.DataFrame(rows)
    sub_file = "submission_hybrid_phase2.csv"
    sub_df.to_csv(sub_file, index=False)

    with zipfile.ZipFile("submission_hybrid_phase2.zip", "w") as zf:
        zf.write(sub_file)

    print(f" DONE! File: submission_hybrid_phase2.zip ({len(sub_df)} users)")
    print(sub_df.head())

if __name__ == "__main__":
    phase_2_production()